## dataset

In [1]:
!pip install tqdm fastparquet

In [1]:
import requests
import yaml
import getpass
# import zstandard as zstd
import pandas as pd
import json
import io

from typing import Dict, List, Any
from tqdm import tqdm

In [2]:
df_queries = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_examples.parquet')
df_queries = df_queries[df_queries["product_locale"] == "us"]


In [3]:
df_queries.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train


In [4]:
unique_queries = df_queries["query"].drop_duplicates()

In [5]:
print([len(df_queries), len(unique_queries)])

[1818825, 97345]


In [6]:
unique_queries.head(5)

0                                revent 80 cfm
16                !awnmower tires without rims
32                !qscreen fence without holes
149    # 10 self-seal envelopes without window
189                  # 2 pencils not sharpened
Name: query, dtype: object

In [7]:
import os

QUERY_SAMPLE_FILE = 'random_queries.json'

if os.path.exists(QUERY_SAMPLE_FILE):
    print(f"{QUERY_SAMPLE_FILE} already exists -- keeping it, not redrawing")
else:
    sample = unique_queries.sample(n=1000, random_state=42)
    sample.to_frame(name='query').to_json(
        QUERY_SAMPLE_FILE, orient='records', indent=1, force_ascii=False
    )

random_queries.json already exists -- keeping it, not redrawing


this is actually incorrect. or oversimplification. you need not random queries but carefuly tailored selection.

In [8]:
df_query_sample = pd.read_json(QUERY_SAMPLE_FILE)

random_queries = df_query_sample['query']
len(random_queries)

1000

In [10]:
df_products = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet')
df_products = df_products[df_products["product_locale"] == "us"]
df_products.head(5)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
167168,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,None,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us
167169,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,None,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us
167170,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,None,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us
167171,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,None,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
167172,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,None,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us


In [11]:
df_random_queries = df_queries[
    df_queries["query"].isin(random_queries)
]

In [12]:
df = pd.merge(
    df_random_queries,
    df_products,
    how='left',
    left_on=['product_locale','product_id'],
    right_on=['product_locale', 'product_id']
)
df.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
0,3042,$5 items,100,B079HXXP4T,us,I,1,1,test,"Soft Scrub In-Tank Toilet Cleaner Duo-Cubes, A...",None,"Helps fight toilet ring, hard water, and limes...",Soft Scrub,Alpine Fresh
1,3043,$5 items,100,B07DZYGDS3,us,E,1,1,test,"Gillette Fusion5 Razors for Men, 1 Gillette Ra...",None,REFILLS FIT ALL GILLETTE 5-BLADE RAZOR HANDLES...,Gillette,None
2,3044,$5 items,100,B07HY9DC4N,us,E,1,1,test,"Summer's Eve Cleansing Cloths, Blissful Escape...",None,Summer's Eve Feminine Cleansing Wipes are safe...,Summer's Eve,None
3,3045,$5 items,100,B07M77RB97,us,E,1,1,test,BIC Flex 5 Hybrid Men's 5-Blade Disposable Raz...,None,"5 long lasting, flexible blades individually a...",BIC,Black
4,3046,$5 items,100,B07NTWYGJX,us,I,1,1,test,"6PCS Dual Heads Blackhead Remover, Pimple Come...",<b>About Our Factory:</b><br /> ✿✿Our factory ...,"♥ 【Dual Heads REMOVER】: 6PCS dual heads tools,...",USCOLOR,None


In [13]:
len(df)

18727

## index

In [44]:
SEARCH_INDEX = 'http://localhost:9200/megacities'

In [45]:
idx = requests.put(
    SEARCH_INDEX,
    json={
        "mappings": {
            "properties": {
                "name": {
                    "type": "text"
                },
                "description": {
                    "type": "text"
                }
            }
        }
    }
)
idx.json()

{'acknowledged': True, 'shards_acknowledged': True, 'index': 'megacities'}

In [46]:
def index_record(id, name, description):
    if id and name and description:
        try:
            return requests.post(
                f"{SEARCH_INDEX}/_doc/{id}",
                json={
                    'name': name,
                    'description': description    
                }
            )
        except:
            pass

In [47]:
for index, row in tqdm(df.iterrows(), total=len(df)):
    _ = index_record(row['example_id'], row['product_title'], row['product_description'])

100%|█████████████████████████████████████████████████████████████| 18727/18727 [00:20<00:00, 897.87it/s]


In [48]:
response = requests.post(
    f"{SEARCH_INDEX}/_search",
    json={
        "size": 0,
        "track_total_hits": True
    }
)
response.json()

{'took': 37,
 'timed_out': False,
 'terminated_early': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 9295, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

In [49]:
def search_query(query='#$query##'):
    return {
      "size": 10,  
      "query": {
        "multi_match": {
          "query": query,
          "fields": [f"name", "description"]
        }
      }
    }
    
def search(query):
    response = requests.post(
        f"{SEARCH_INDEX}/_search",
        json=search_query(query)
    )
    return response.json()

In [50]:
search('dinosaur')

{'took': 75,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 31, 'relation': 'eq'},
  'max_score': 13.017879,
  'hits': [{'_index': 'megacities',
    '_id': '1975317',
    '_score': 13.017879,
    '_source': {'name': 'Baby Dinosaur Balloon Set for Birthday Decor - 38 Inch, Pack of 4, 4D Dinosaur Foil Balloon | Kids Dinosaur Party Decorations | Dinosaur Balloons for Birthday Party | Dinosaur Birthday Party Supplies',
     'description': '<p><b>Are you a dinasaur fanatic?</b></p><p>Looking for a dinosaur theme party decorations for your kid</p> <p>We have got you covered with these beautiful and gigantic <b>Baby Dinosaur Foil Balloons </b> in two different style of dinosaur party balloons </p> <p>This dinosaur balloon kit is simply beautiful, gorgeous color to your dinosaur balloon birthday party as a backdrop for photos Booth.</p> <p>Configure it any way you wish; there are 1000s of ways to use your kit to build you

## Quepid

### init

In [51]:
!docker compose run quepid-api-quepid bin/rake db:migrate
!docker compose run quepid-api-quepid bin/rake db:seed
!docker compose run quepid-api-quepid bundle exec thor user:create -a admin@example.com "Admin User" supersecret

[+] run 2/2
 ✔ Container es-test                          Running                       0.0s
 ✔ Container judge-student-quepid-api-mysql-1 Running                       0.0s
Container judge-student-quepid-api-mysql-1 Waiting 
Container es-test Waiting 
Container judge-student-quepid-api-mysql-1 Healthy 
Container es-test Healthy 
Container judge-student-quepid-api-quepid-run-3bc67dfe88db Creating 
Container judge-student-quepid-api-quepid-run-3bc67dfe88db Created 

!!! RubyLLM's legacy acts_as API is deprecated and will be removed in RubyLLM 2.0.0. Please consult the migration guide at https://rubyllm.com/upgrading-to-1-7/

WARN[0000] Found orphan containers (judge-student-quepid-api-quepid-run-3bc67dfe88db) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up. 
[+] run 2/2
 ✔ Container es-test                          Running                       0.0s
 ✔ Container judge-student-quepid-api

In [52]:
import re

out = !docker compose run quepid-api-quepid bundle exec thor user:add_api_key admin@example.com

found = re.search(r"[0-9a-f]{64}", "\n".join(out))
if not found:
    raise RuntimeError("no API key in output:\n" + "\n".join(out))

QUEPID_TOKEN = found.group()

In [53]:
QUEPID_TOKEN

'aac8ec04cb133d585c6a10cae879d7cdc1718ac056d7996e1dad4d8022a5a382'

### !!

In [54]:
AUTH = {
    "Authorization": f"Bearer {QUEPID_TOKEN}"
}

In [55]:
team = requests.post(
    'http://localhost:8081/api/teams/', 
    headers = AUTH,
    json={
        "name": "Justice Department"
    }   
)
team = team.json()

In [56]:
team

{'id': 1,
 'name': 'Justice Department',
 'created_at': '2026-09-10T19:34:22.555Z',
 'updated_at': '2026-09-10T19:34:22.555Z'}

In [57]:
endpoint = requests.post(
    'http://localhost:8081/api/search_endpoints/', 
    headers = AUTH,
    json={
        "name": "Megacities",
        "endpoint_url": f"http://quepid-api-elasticsearch:9200/{SEARCH_INDEX}/_search",
        "search_engine": "es",
        "api_method": "POST",
        "proxy_requests": 1,   
    }   
)
endpoint = endpoint.json()

In [58]:
print(endpoint)

{'id': 1, 'name': 'Megacities', 'owner': 1, 'search_engine': 'es', 'endpoint_url': 'http://quepid-api-elasticsearch:9200/http://localhost:9200/megacities/_search', 'api_method': 'POST', 'custom_headers': None, 'archived': 0, 'created_at': '2026-09-10T19:34:26.925Z', 'updated_at': '2026-09-10T19:34:26.925Z', 'basic_auth_credential': None, 'mapper_code': None, 'proxy_requests': 1, 'options': None, 'requests_per_minute': None, 'test_query': None}


In [59]:
# list scorers
scorers = requests.get(
    'http://localhost:8081/api/scorers/', 
    headers = AUTH
)
print({s['id']: s['name'] for s in scorers.json()['items']})

{1: 'nDCG@10', 2: 'DCG@10', 3: 'CG@10', 4: 'P@10', 5: 'AP@10', 6: 'RR@10', 7: 'ERR@10'}


In [60]:
# Two books, one rater each, and that is the whole point of this layout.
#
# Quepid syncs a book's judgements down to its cases -- and to *every* case
# attached to that book: RatingsManager#sync_ratings_for_query_doc_pair does
# `@book.cases.each`, with no way to narrow it on this version. Worse, it
# blends every rater on a pair into one number via
# calculate_rating_from_judgements (top three, the agreed value if they agree,
# otherwise the minimum). So two cases hanging off one book would end up with
# identical, averaged ratings, and the human/AI comparison would disappear
# exactly when both sides had rated.
#
# One rater per book avoids it: with a single rater the blend is a no-op, and
# each case gets its own side of the experiment.
def make_book(name):
    r = requests.post(
        'http://localhost:8081/api/books/',
        headers = AUTH,
        json={
            "name": name,
            # ESCI's four labels in descending relevance, so a judgement can be
            # compared with esci_label directly. A book with no scale renders no
            # rating buttons at all.
            "scale": [0, 1, 2, 3],
            "scale_with_labels": {
                "0": "Irrelevant",
                "1": "Complement",
                "2": "Substitute",
                "3": "Exact",
            },
        }
    )
    r.raise_for_status()
    return r.json()


def make_case(name, book):
    r = requests.post(
        'http://localhost:8081/api/case/',
        headers = AUTH,
        json={
            "name": name,
            "scorer_id": 1,
            "book_id": book['id'],
            "search_endpoint_id": endpoint.get('id'),
            "search_query": json.dumps(search_query()),
        }
    )
    r.raise_for_status()
    return r.json()


In [61]:
# the humans' side.
# ESCI's labels go in as judgements, and no AI
# judge is ever attached to this book, so nothing overwrites them.
book_esci = make_book("Australo-Hungaria")
case_esci = make_case("Australo-Hungaria", book_esci)

# Neo-Chicago -- the machine's side. The same pairs, but no labels: every
# judgement in this book comes from the AI judge.
book_ai = make_book("Neo-Chicago")
case_ai = make_case("Neo-Chicago", book_ai)

# {c['name']: {"case": c['id'], "book": c['book_id']} for c in (case_esci, case_ai)}

In [62]:
book_esci, case_esci

({'scale': [0, 1, 2, 3],
  'scale_with_labels': {'0': 'Irrelevant',
   '1': 'Complement',
   '2': 'Substitute',
   '3': 'Exact'},
  'id': 1,
  'name': 'Australo-Hungaria',
  'created_at': '2026-09-10T19:34:47.952Z',
  'updated_at': '2026-09-10T19:34:47.952Z',
  'support_implicit_judgements': 0,
  'show_rank': 0,
  'owner_id': 1,
  'export_job': None,
  'import_job': None,
  'populate_job': None,
  'archived': 0,
  'scoring_guidelines': None},
 {'id': 1,
  'case_name': 'Australo-Hungaria',
  'last_try_number': 1,
  'owner': 1,
  'archived': 0,
  'scorer_id': 1,
  'created_at': '2026-09-10T19:34:47.967Z',
  'updated_at': '2026-09-10T19:34:47.967Z',
  'book_id': 1,
  'public': None,
  'options': None,
  'nightly': 1})

In [63]:
# A book's queries are the distinct query_texts of its query/doc pairs, and a
# pair must carry a doc -- so the ground truth goes in whole, one pair per df
# row. doc_id is example_id, matching what index_record used as the
# Elasticsearch _id, so the book lines up with what a search on myindex returns
# (and joins straight back onto df, esci_label included).
df_pairs = df[df['product_title'].notna() & df['product_description'].notna()]

pairs = [
    {
        "query_text": row['query'],
        "doc_id": str(row['example_id']),
        "document_fields": {
            "name": row['product_title'],
            "description": row['product_description'],
        },
        # Nothing in query_doc_pairs holds a foreign id, so the ESCI query_id
        # travels in options. Deliberately not esci_label: the judge reads the
        # document, and the label is recoverable from doc_id anyway.
        "query_options": {"esci_query_id": int(row['query_id'])},
    }
    for _, row in df_pairs.iterrows()
]
print(f"{df_pairs['query'].nunique()} queries, {len(pairs)} pairs")


# Batched: a pair is identified by (query_text, doc_id), so re-running this
# cell adds nothing and a batch that fails can simply be sent again.
def load_pairs(book):
    written = {"created": 0, "skipped": 0}
    for i in tqdm(range(0, len(pairs), 500), desc=book['name']):
        r = requests.post(
            f"http://localhost:8081/api/books/{book['id']}/query_doc_pairs/",
            headers = AUTH,
            json=pairs[i:i + 500]
        )
        r.raise_for_status()
        for k, v in r.json().items():
            written[k] += v
    return written


# Both books get the identical corpus. They differ only in who judges it.
{book['name']: load_pairs(book) for book in (book_esci, book_ai)}

945 queries, 9295 pairs


Neo-Chicago: 100%|███████████████████████████████████████████████████████| 19/19 [00:01<00:00, 16.67it/s]


{'Australo-Hungaria': {'created': 9295, 'skipped': 0},
 'Neo-Chicago': {'created': 9295, 'skipped': 0}}

In [64]:
# The labels, into Babilon only. A pair carries no rating: the rating is a
# *judgement*, one row per (rater, pair), so ESCI's ground truth goes in as
# judgements of ours. Mega-City One deliberately gets none of this -- its
# judgements are the AI judge's work, and that is what makes the two cases
# comparable rather than contaminated.
ESCI_RATING = {"I": 0, "C": 1, "S": 2, "E": 3}   # the book's scale, set above

labels = [
    {
        "query_text": row['query'],
        "doc_id": str(row['example_id']),
        "rating": ESCI_RATING[row['esci_label']],
        "explanation": f"ESCI ground truth: {row['esci_label']}",
    }
    for _, row in df_pairs.iterrows()
]

# Identity is (rater, pair), so re-running updates rather than duplicating --
# unlike the pairs above, which are skipped. Rows naming a pair the book does
# not hold come back as "unknown" instead of failing the batch, so a non-zero
# count there means labels went nowhere and is worth looking at.
written = {"created": 0, "updated": 0, "unchanged": 0, "unknown": 0}
for i in tqdm(range(0, len(labels), 500)):
    r = requests.post(
        f"http://localhost:8081/api/books/{book_esci['id']}/judgements/",
        headers = AUTH,
        json=labels[i:i + 500]
    )
    r.raise_for_status()
    for k, v in r.json().items():
        written[k] += v

written

100%|████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00, 14.08it/s]


{'created': 9295, 'updated': 0, 'unchanged': 0, 'unknown': 0}

In [65]:
# Both cases now need their queries. Attaching a book to a case copies nothing:
# a book holds query/doc *pairs*, a case holds *queries*, and cases.book_id is
# the only thing joining the two tables. A case with no queries runs nothing
# against the search endpoint and scores nothing.
df_case_queries = df_pairs[['query_id', 'query']].drop_duplicates('query')


def load_case_queries(case):
    # Read first, so the cell is re-runnable: nothing de-duplicates case
    # queries server-side, and there is no bulk endpoint for them either --
    # unlike pairs and judgements, this is one POST per query.
    have = requests.get(
        f"http://localhost:8081/api/query/{case['id']}/?limit=100000",
        headers = AUTH
    )
    have.raise_for_status()
    have = {q['query_text'] for q in have.json()['items']}

    added = 0
    for _, row in tqdm(df_case_queries.iterrows(),
                       total=len(df_case_queries), desc=case['case_name']):
        if row['query'] in have:
            continue
        q = requests.post(
            f"http://localhost:8081/api/query/{case['id']}/",
            headers = AUTH,
            # The ESCI query_id rides along here too, so a case query joins back
            # to df and to the book's pairs on something other than the text.
            json={"query_text": row['query'],
                  "query_options": {"esci_query_id": int(row['query_id'])}}
        )
        q.raise_for_status()
        added += 1
    return {"added": added, "already there": len(have)}


{case['case_name']: load_case_queries(case) for case in (case_esci, case_ai)}

Neo-Chicago: 100%|████████████████████████████████████████████████████| 945/945 [00:04<00:00, 230.38it/s]


{'Australo-Hungaria': {'added': 945, 'already there': 0},
 'Neo-Chicago': {'added': 945, 'already there': 0}}

### AI Judge

The judge does not talk to OpenAI directly -- it goes through the LiteLLM proxy
(`quepid-api-litellm`, added to `docker-compose.yml`), which is an
OpenAI-compatible gateway. Two reasons that is worth the extra hop:

- **The provider key never reaches Quepid.** Quepid stores `llm_key` in its own
  database; what it gets here is a LiteLLM client key scoped to one model, and
  the real `OPENAI_API_KEY` stays in `.env-private` on the proxy.
- **Responses are cached in S3.** Judging replays near-identical prompts across
  re-runs, and an exact-match hit costs nothing. See `cache_params` in
  `data/litellm/config.yaml`.

Swapping the judge onto a different model or provider is then a config edit on
the proxy rather than a change here.

In [66]:
# Not an OpenAI key: a LiteLLM *client* key, out of the `client_keys` block in
# data/litellm/config.yaml. Hardcoded on purpose -- these are local-dev keys
# that exist only inside this compose stack, and the value has to match the
# config exactly, so prompting for it would only invite typos.
#
# LiteLLM has no built-in config option for client keys; the block is read by
# data/litellm/custom_auth.py, which the proxy loads as its auth hook. The key
# is scoped there to gpt-4o and gpt-4o-mini, so a judge misconfigured onto some
# other model gets a 403 rather than a bill.
llm_key = "sk-quepid-judge-local-dev"

# To bypass the proxy and hit OpenAI directly instead, swap this for a real
# provider key and set llm_service_url to https://api.openai.com below:
# llm_key = getpass.getpass("LLM API key: ")

In [67]:
# The rubric the judge actually rates against. Quepid's default system prompt
# (AiJudgesController::DEFAULT_SYSTEM_PROMPT) is a generic relevance ladder --
# 0 irrelevant, 1 somewhat, 2 mostly, 3 perfectly relevant -- with no notion of
# Substitute or Complement. That is a different question from the one ESCI asks,
# and the disagreement it causes is systematic rather than noisy: measured over
# 2,424 pairs it demoted 599 of ESCI's Exacts to 2 and scattered its Substitutes
# down to 0/1, running a full 0.8 of a point harsh (AI mean 1.57 vs 2.36).
#
# LlmService reads this off judgement.user.system_prompt -- a column on users,
# capped at 4000 characters -- so it is a top-level key here, NOT part of
# judge_options.
#
# The JSON block at the end is mandatory and the spelling of "judgment" is
# load-bearing: get_llm_response parses the reply and reads exactly
# parsed_content['explanation'] and parsed_content['judgment'] (American
# spelling, no 'e', unlike 'judgement' everywhere else in Quepid). Any other key
# leaves rating nil, whereupon RunJudgeJudyJob calls mark_unrateable on every
# pair and the whole run yields nothing.
ESCI_SYSTEM_PROMPT = """\
You are evaluating results from a product search engine. For each query you are given one document, and you assign a judgment on the ESCI scale of 0 to 3:

- 3 (Exact): the product is relevant to the query and satisfies every specification the query states. If the query names an attribute -- size, colour, quantity, material, model, brand, gender, compatibility -- the product must match it.
- 2 (Substitute): the product fails some stated specification but is a functional substitute a shopper could reasonably buy instead. Same job, different attribute. A fleece for "sweater"; a 16oz bottle for "24oz water bottle".
- 1 (Complement): the product does not answer the query itself, but is bought and used together with something that would. Track pants for "running shoes"; ink for "printer".
- 0 (Irrelevant): the product is unrelated, or fails a central aspect of the query. Socks for "running shoes".

Judge only what the query asks. A query that states no attributes is satisfied by any product of the right type -- rate it 3, not 2. Do not lower the judgment because the document is short, or because you cannot confirm a detail the query never asked about. Reserve 1 for genuine companion products, not for weak relevance: a product that is merely a poor match is 0.

The response should be in the following JSON format:
{
  "explanation": "Your detailed reasoning behind the judgment",
  "judgment": <numeric value>
}

Here is an example:
User:
Query: stainless steel water bottle 32 oz

doc1:
  name: Hydro Flask 32 oz Wide Mouth Stainless Steel Water Bottle
  description: Vacuum insulated 32 ounce stainless steel bottle.
Assistant:
{
  "explanation": "Matches the product type, the stated material and the stated 32 oz capacity.",
  "judgment": 3
}

User:
Query: stainless steel water bottle 32 oz

doc1:
  name: Nalgene Tritan 32 oz Wide Mouth Bottle
  description: BPA-free plastic 32 ounce bottle.
Assistant:
{
  "explanation": "A 32 oz water bottle and a usable substitute, but plastic rather than the stainless steel the query specifies.",
  "judgment": 2
}

User:
Query: stainless steel water bottle 32 oz

doc1:
  name: Bottle Cleaning Brush Set
  description: Long handled brushes for cleaning narrow bottles and straws.
Assistant:
{
  "explanation": "Not a water bottle, but bought alongside one to clean it.",
  "judgment": 1
}

User:
Query: stainless steel water bottle 32 oz

doc1:
  name: Stainless Steel Cutlery Set, 20 Piece
  description: Forks, knives and spoons in brushed stainless steel.
Assistant:
{
  "explanation": "Shares the material but is not a water bottle and serves no part of the need.",
  "judgment": 0
}
"""

JUDGE_OPTIONS = {
    "llm_provider": "openai",
    "llm_service_url": "http://quepid-api-litellm:4000",
    "llm_model": "gpt-4o",
    "llm_timeout": 30,
    "llm_api_version": "",
}

JUDGE_NAME = "Dredd"

# Look before writing, because a judge has no natural key: a bare POST on a
# second run makes a *second* Dredd rather than updating the first, and
# run_judge_judy would then offer you two identical judges to pick between.
# Matching on name keeps this cell re-runnable the way cell 39 is.
existing = {
    j['name']: j
    for j in requests.get(
        'http://localhost:8081/api/ai_judges/?limit=1000', headers = AUTH
    ).json()['items']
}

payload = {
    "name": JUDGE_NAME,
    "llm_key": llm_key,
    "system_prompt": ESCI_SYSTEM_PROMPT,
    "judge_options": JUDGE_OPTIONS,
}

if JUDGE_NAME in existing:
    judge = requests.put(
        f"http://localhost:8081/api/ai_judges/{existing[JUDGE_NAME]['id']}/",
        headers = AUTH,
        json=payload
    )
else:
    judge = requests.post(
        'http://localhost:8081/api/ai_judges/',
        headers = AUTH,
        json=payload
    )

judge.raise_for_status()
judge = judge.json()

# Changing the prompt does not touch judgements already made under the old one.
# Those were answers to a different question, so mixing them into the teacher
# signal blends two rubrics -- clear them deliberately before re-running Dredd:
#   requests.delete(f"http://localhost:8081/api/books/{book_ai['id']}/judgements/",
#                   headers=AUTH, params={"user_id": judge['id']})
judge['id'], judge['name'], len(judge['system_prompt']), judge['judge_options']


(2,
 'Dredd',
 2650,
 {'llm_provider': 'openai',
  'llm_service_url': 'http://quepid-api-litellm:4000',
  'llm_model': 'gpt-4o',
  'llm_timeout': 30,
  'llm_api_version': ''})

In [68]:
# Mega-City One only. Sharing a team is not enough to make a judge usable on a
# book -- run_judge_judy picks from book.ai_judges, which is this join -- and
# attaching it to Babilon as well is precisely what this layout exists to
# prevent: its judgements would be blended into Babilon's ratings alongside
# ESCI's, and both cases would end up saying the same thing.
attached = requests.post(
    f"http://localhost:8081/api/ai_judges/{judge['id']}/books/",
    headers = AUTH,
    json={"book_id": book_ai['id']}
)
attached.raise_for_status()

# Idempotent, so re-running is a no-op -- unlike the cell above, which has no
# natural key and would make a second judge.
[(b['id'], b['name']) for b in requests.get(
    f"http://localhost:8081/api/ai_judges/{judge['id']}/books/", headers = AUTH
).json()]

[(2, 'Neo-Chicago')]

### Manually start judging

In [42]:
# Nothing here starts the judging run: run_judge_judy is an HTML route that
# enqueues RunJudgeJudyJob, and neither Quepid's API nor this one exposes it.
# Open Mega-City One's book, pick the judge, and let it work through the pairs.
# When the run finishes it enqueues UpdateCaseJob, which fills case Mega-City
# One's ratings from those judgements by itself -- Babilon is untouched, having
# no judge and nothing to enqueue.
print(f"http://localhost:3000/books/{book_ai['id']}/judgement_stats")

http://localhost:3000/books/2/judgement_stats


### Also poke baseline case

In [69]:
# The judgements loaded above live in the *book*. The case reads *ratings*, a
# separate table, and nothing has copied one into the other: RatingsManager is
# only ever reached from UpdateCaseRatingsJob (Quepid's judging screen, one pair
# at a time) or UpdateCaseJob. The bulk judgements endpoint writes MySQL through
# Django and so enqueues neither -- which is why Neo-Chicago fills itself in for
# free at the end of RunJudgeJudyJob while Australo-Hungaria stays empty.
#
# This is the one endpoint whose only job is that copy. Note the port: it is
# Quepid's own Rails API, not the 8081 one the rest of this notebook talks to.
# Same bearer token either way -- Api::ApiController authenticates off the same
# api_keys row that thor minted in cell 25.
QUEPID_URL = 'http://localhost:3000'

refresh = requests.put(
    f"{QUEPID_URL}/api/books/{book_esci['id']}/cases/{case_esci['id']}/refresh",
    headers = AUTH,
    # Rails reads these off params, so they are query string rather than body;
    # the body is sent empty, matching what bookSvc.refreshCaseRatingsFromBook
    # does. deserialize_bool_param parses the strings, so 'false' is honest.
    params={
        # False on purpose. sync_judgements_to_ratings returns early on a query
        # the case does not already hold, and cell 39 put all 945 in -- so this
        # rates the ESCI pairs and silently ignores the search-result pairs that
        # PopulateBookJob added, which is exactly what we want. True would start
        # inventing case queries out of query_doc_pairs.
        "create_missing_queries": "false",
        # Inline, so the counts below are real. See the note on timing.
        "process_in_background": "false",
    },
    json={},
    # sync_ratings_for_case walks *every* pair in the book -- 28,935 of them
    # here, not just the 9,295 judged ones -- doing a query lookup, a save and a
    # Turbo broadcast for each. That is a long request, so no default timeout.
    timeout=None,
)
refresh.raise_for_status()
refresh.json()

{'queries_created': 0, 'ratings_created': 9295, 'process_in_background': False}

In [71]:
# The judgements loaded above live in the *book*. The case reads *ratings*, a
# separate table, and nothing has copied one into the other: RatingsManager is
# only ever reached from UpdateCaseRatingsJob (Quepid's judging screen, one pair
# at a time) or UpdateCaseJob. The bulk judgements endpoint writes MySQL through
# Django and so enqueues neither -- which is why Neo-Chicago fills itself in for
# free at the end of RunJudgeJudyJob while Australo-Hungaria stays empty.
#
# This is the one endpoint whose only job is that copy. Note the port: it is
# Quepid's own Rails API, not the 8081 one the rest of this notebook talks to.
# Same bearer token either way -- Api::ApiController authenticates off the same
# api_keys row that thor minted in cell 25.
QUEPID_URL = 'http://localhost:3000'

refresh = requests.put(
    f"{QUEPID_URL}/api/books/{book_ai['id']}/cases/{case_ai['id']}/refresh",
    headers = AUTH,
    # Rails reads these off params, so they are query string rather than body;
    # the body is sent empty, matching what bookSvc.refreshCaseRatingsFromBook
    # does. deserialize_bool_param parses the strings, so 'false' is honest.
    params={
        # False on purpose. sync_judgements_to_ratings returns early on a query
        # the case does not already hold, and cell 39 put all 945 in -- so this
        # rates the ESCI pairs and silently ignores the search-result pairs that
        # PopulateBookJob added, which is exactly what we want. True would start
        # inventing case queries out of query_doc_pairs.
        "create_missing_queries": "false",
        # Inline, so the counts below are real. See the note on timing.
        "process_in_background": "false",
    },
    json={},
    # sync_ratings_for_case walks *every* pair in the book -- 28,935 of them
    # here, not just the 9,295 judged ones -- doing a query lookup, a save and a
    # Turbo broadcast for each. That is a long request, so no default timeout.
    timeout=None,
)
refresh.raise_for_status()
refresh.json()

{'queries_created': 0, 'ratings_created': 0, 'process_in_background': False}

## judge vs ground truth

Not the two cases' nDCG scores. Those are not comparable to each other: the
scorer maps an unrated document to 0, and the two cases have very different
rating coverage in the top 10 -- Australo-Hungaria only ever rates the ESCI
pairs, while Dredd works through everything in Neo-Chicago's book, including
the search-result pairs `PopulateBookJob` added. The ideal DCG diverges too,
being summed over `ideal.length` rather than 10. Whatever the difference
between the two scores turns out to be, coverage explains more of it than
judge quality does.

So the teacher is measured per pair instead, on the pairs where ESCI and the
judge both have an opinion. Joining on `(query_text, doc_id)` makes coverage
drop out entirely.

Agreement between ESCI ground truth and the AI judge, over the intersection of
what they have both rated. Re-runnable while Dredd is still working: it just
widens the sample.

In [ ]:
import numpy as np


def fetch_all(url, params=None, page=5000):
    """Both endpoints are @paginate'd, so walk limit/offset to the end."""
    out, offset = [], 0
    while True:
        p = dict(params or {})
        p.update(limit=page, offset=offset)
        r = requests.get(url, headers=AUTH, params=p)
        r.raise_for_status()
        body = r.json()
        out.extend(body['items'])
        if len(out) >= body['count'] or not body['items']:
            return out
        offset += page


# Filtered by rater. Nothing else judges this book today, but SelectionStrategy
# allows three raters per pair, so leaving it unfiltered would silently blend in
# anybody else's verdicts the day a second judge is attached.
ai_judgements = fetch_all(
    f"http://localhost:8081/api/books/{book_ai['id']}/judgements/",
    {"user_id": judge['id']}
)

# A judgement carries only query_doc_pair_id, and the same (query, doc) has
# different pair ids in the two books -- they were inserted separately. So the
# join back to ESCI has to go through the pairs to recover query_text/doc_id.
book_pairs = fetch_all(
    f"http://localhost:8081/api/books/{book_ai['id']}/query_doc_pairs/"
)

pair_map = pd.DataFrame([
    {"query_doc_pair": p['id'], "query": p['query_text'], "doc_id": p['doc_id']}
    for p in book_pairs
])

df_ai = pd.DataFrame([
    {"query_doc_pair": j['query_doc_pair'], "ai": j['rating'],
     "unrateable": j['unrateable']}
    for j in ai_judgements
]).merge(pair_map, on="query_doc_pair", how="left")

# RunJudgeJudyJob calls mark_unrateable whenever the model came back with no
# parseable rating, and those rows carry rating NULL. That is an absence of an
# opinion, not a 0, and averaging it in as one would flatter nothing.
df_ai = df_ai[(df_ai['unrateable'] == 0) & df_ai['ai'].notna()]

# The human side needs no fetching: df_pairs already holds it, and cell 37 wrote
# doc_id as str(example_id), so it joins straight back.
truth = df_pairs[['query', 'example_id', 'esci_label']].copy()
truth['doc_id'] = truth['example_id'].astype(str)
truth['human'] = truth['esci_label'].map(ESCI_RATING)

# Inner join: the intersection is the only place agreement is defined. The pairs
# the judge rated that ESCI never labelled are dropped here, and counted below.
cmp = df_ai.merge(truth[['query', 'doc_id', 'human', 'esci_label']],
                  on=['query', 'doc_id'], how='inner')
cmp['ai'] = cmp['ai'].astype(int)
print(f"{len(cmp)} of {len(df_ai)} AI judgements land on ESCI-labelled pairs")

delta = (cmp['ai'] - cmp['human']).abs()
print(f"exact agreement : {(delta == 0).mean():.3f}")
print(f"within +/-1     : {(delta <= 1).mean():.3f}")
print(f"mean abs error  : {delta.mean():.3f}")
print(f"AI mean {cmp['ai'].mean():.2f}  vs  human mean {cmp['human'].mean():.2f}")

# Quadratic weighted kappa: the right statistic for an ordinal scale, since it
# punishes an Exact-called-Irrelevant far harder than an Exact-called-Substitute,
# and discounts the agreement you would get by chance from the label imbalance.
# Hand-rolled because neither sklearn nor scipy is in this venv.
k = 4
O = pd.crosstab(cmp['human'], cmp['ai']) \
      .reindex(index=range(k), columns=range(k), fill_value=0).values.astype(float)
O /= O.sum()
E = np.outer(O.sum(1), O.sum(0))
w = (np.arange(k)[:, None] - np.arange(k)[None, :]) ** 2 / (k - 1) ** 2
print(f"quadratic weighted kappa: {1 - (w * O).sum() / (w * E).sum():.3f}")
print(f"spearman: {cmp['ai'].rank().corr(cmp['human'].rank()):.3f}")

# Where the disagreement actually sits, which the scalars above cannot show.
print("\nconfusion (rows=ESCI, cols=AI):")
print(pd.crosstab(cmp['esci_label'], cmp['ai']).reindex(['E', 'S', 'C', 'I']))

# ESCI's usual binarisation, and the one that matters if the student is trained
# on relevant/not rather than on the full four-point scale.
print(f"\nbinarised (>=2 relevant): "
      f"{((cmp['human'] >= 2) == (cmp['ai'] >= 2)).mean():.3f}")

## train student

In [72]:
QUEPID_API = 'http://localhost:8081/api'


In [73]:
def fetch_all(url, params=None, page=5000):
    out, offset = [], 0
    while True:
        p = dict(params or {})
        p.update(limit=page, offset=offset)
        r = requests.get(url, headers=AUTH, params=p)
        r.raise_for_status()
        body = r.json()
        out.extend(body['items'])
        if len(out) >= body['count'] or not body['items']:
            return out
        offset += page


def fetch_book_pairs(book, rater_id=None, judged_only=False):
    """Every query/doc pair in `book`, one row each, with its rating alongside.

    rater_id filters the judgements to one rater. Leave it None only on a
    single-rater book: SelectionStrategy allows three raters per pair and the
    merge below is one row per (pair, rater), so an unfiltered multi-rater book
    repeats a pair once per verdict.
    """
    pairs = fetch_all(f"{QUEPID_API}/books/{book['id']}/query_doc_pairs/")

    df_book = pd.DataFrame([{
        "pair_id": p['id'],
        "query": p['query_text'],
        "doc_id": p['doc_id'],
        # Where the pair came from: PopulateBookJob writes the rank the engine
        # returned the doc at; the pairs loaded in cell 38 carry None.
        "position": p['position'],
        "information_need": p['information_need'],
        # The ESCI query_id that rode along in query_options.
        "esci_query_id": (p['query_options'] or {}).get('esci_query_id'),
        # document_fields is TEXT holding JSON server-side, but the schema's
        # resolver parses it -- so name/description for the pairs we loaded,
        # and whatever the mapper produced for the ones Quepid added itself.
        **(p['document_fields'] or {}),
    } for p in pairs])

    judgements = fetch_all(
        f"{QUEPID_API}/books/{book['id']}/judgements/",
        {"user_id": rater_id} if rater_id else None,
    )
    df_j = pd.DataFrame(
        [{
            "pair_id": j['query_doc_pair'],
            "rater_id": j['user'],
            "rating": j['rating'],
            # mark_unrateable rows carry rating NULL: an absence of an opinion,
            # not a 0.
            "unrateable": bool(j['unrateable']),
            "explanation": j['explanation'],
        } for j in judgements],
        columns=["pair_id", "rater_id", "rating", "unrateable", "explanation"],
    )

    # Left join, so pairs nobody has rated yet survive with rating NaN -- worth
    # seeing while a judging run is still in flight rather than silently gone.
    out = df_book.merge(df_j, on="pair_id", how="left")
    if judged_only:
        out = out[out['rating'].notna()
                  & ~out['unrateable'].fillna(False).astype(bool)]
    return out.reset_index(drop=True)


In [74]:
df_teacher = fetch_book_pairs(book_ai, rater_id=judge['id'])   # Dredd's verdicts
df_truth   = fetch_book_pairs(book_esci)

In [75]:
df_teacher.head(5)

,pair_id,query,doc_id,position,information_need,esci_query_id,title,rater_id,rating,unrateable,explanation
0,9302,'grinch ornaments',4014,4,,134,Kurt Adler Dr. Seuss The Grinch Santa Grinch O...,2,3.0,False,The product is an ornament featuring the Grinc...
1,9303,'grinch ornaments',4017,1,,134,Kurt Adler Grinch Santa and Dog Max Holiday Or...,2,3.0,False,The document describes a set of ornaments feat...
2,9304,'grinch ornaments',4018,3,,134,Johnson Smith Co. - Kurt S Adler INC Grinch wi...,2,3.0,False,The product is a Grinch Christmas tree ornamen...
3,9305,'grinch ornaments',4019,8,,134,24Pcs Christmas Balls Ornaments for Xmas Tree ...,2,0.0,False,"The product is a set of Christmas ornaments, w..."
4,9306,'grinch ornaments',4024,2,,134,Dr Seuss Grinch Kurt Adler Holiday Personaliza...,2,3.0,False,"The product is a Grinch ornament, which direct..."


In [79]:
df2 = df_teacher[df_teacher['rating'].notna()]
with open('training_data.jsonl', 'w') as fh:
    for r in df2.itertuples():
        fh.write(json.dumps({"query": r.query,
                             "product_text": r.title,
                             "label": int(r.rating)}) + "\n")


In [ ]:
human labels - better teacher - dspy??

In [81]:
!pip install torch transformers datasets accelerate scikit-learn

Python(5994) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 37.2 MB/s  0:00:03 38.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 37.7 MB/s  0:00:00.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 32.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 36.4 MB/s  0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 33.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 37.8 MB/s  0:00:000.2 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 26.6 MB/s  0:00:000.1 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 27.6 MB/s  0:

In [82]:
import numpy as np

from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [83]:
MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
OUTPUT_DIR = "./dutch-ecommerce-judge"

In [93]:
ID_TO_LABEL = {
    0: "Irrelevant",
    1: "Complement",
    2: "Substitute",
    3: "Exact",
}


LABEL_TO_ID = {
    label: label_id
    for label_id, label in ID_TO_LABEL.items()
}

In [94]:
# Load JSONL.
dataset = load_dataset(
    "json",
    data_files="training_data.jsonl",
    split="train",
)

In [97]:
from datasets import ClassLabel

label_feature = ClassLabel(
    names=list(ID_TO_LABEL.values())
)

dataset = dataset.cast_column("label", label_feature)

dataset = dataset.train_test_split(
    test_size=0.15,
    seed=42,
    stratify_by_column="label",
)

Casting the dataset: 100%|███████████████████████████████| 9111/9111 [00:00<00:00, 2036901.22 examples/s]


In [98]:
from collections import Counter

print(Counter(dataset["train"]["label"]))
print(Counter(dataset["test"]["label"]))

Counter({0: 3384, 3: 3274, 2: 899, 1: 187})
Counter({0: 597, 3: 578, 2: 159, 1: 33})


In [99]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(
        batch["query"],
        batch["product_text"],
        truncation="only_second",
        max_length=256,
    )

In [102]:
print([
    len(ID_TO_LABEL) == 4,
    len(LABEL_TO_ID) == 4
])

[True, True]


In [104]:
print([ID_TO_LABEL, LABEL_TO_ID])

[{0: 'Irrelevant', 1: 'Complement', 2: 'Substitute', 3: 'Exact'}, {'Irrelevant': 0, 'Complement': 1, 'Substitute': 2, 'Exact': 3}]


In [103]:
tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["query", "product_text"],
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,

    # The original reranker has a one-score output head.
    # We replace it with a new randomly initialized three-class head.
    ignore_mismatched_sizes=True,
)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1`.
Loading weights: 100%|██████████████████████████████████████████████| 201/201 [00:00<00:00, 11513.37it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([4])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([4, 384])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [105]:
print(model.config.num_labels)
print(model.config.id2label)
print(model.config.label2id)

4
{0: 'Irrelevant', 1: 'Complement', 2: 'Substitute', 3: 'Exact'}
{'Irrelevant': 0, 'Complement': 1, 'Substitute': 2, 'Exact': 3}


In [106]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
        ),
        "weighted_f1": f1_score(
            labels,
            predictions,
            average="weighted",
        ),
        "weighted_kappa": cohen_kappa_score(
            labels,
            predictions,
            weights="quadratic",
        ),
    }

In [107]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,

    # Suitable for your RTX 4080.
    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",
    seed=42,
)

In [108]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [109]:
trainer.train()

/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Weighted Kappa
1,2.169036,0.986287,0.750549,0.416010,0.698756,0.693733
2,1.740399,0.829005,0.765179,0.448481,0.715139,0.724108
3,1.615226,0.793750,0.767374,0.432761,0.717860,0.735850


Writing model shards: 100%|████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.57it/s]
/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.91it/s]
/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.62it/s]


TrainOutput(global_step=726, training_loss=1.8839894864841598, metrics={'train_runtime': 241.1868, 'train_samples_per_second': 96.324, 'train_steps_per_second': 3.01, 'total_flos': 223580074303488.0, 'train_loss': 1.8839894864841598, 'epoch': 3.0})

In [110]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(trainer.evaluate())

Writing model shards: 100%|████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.76it/s]
/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1,Weighted Kappa
1.615226,0.829005,3,0.765179,0.448481,0.715139,0.724108


{'eval_loss': 0.8290051221847534, 'eval_accuracy': 0.7651792245793709, 'eval_macro_f1': 0.4484814781576015, 'eval_weighted_f1': 0.7151393939912648, 'eval_weighted_kappa': 0.7241082972955367}


Check the actual class performance:

In [111]:
import numpy as np

from sklearn.metrics import classification_report, confusion_matrix

prediction_output = trainer.predict(tokenized_dataset["test"])

y_true = prediction_output.label_ids
y_pred = np.argmax(prediction_output.predictions, axis=1)

class_names = [
    ID_TO_LABEL[i]
    for i in range(len(ID_TO_LABEL))
]

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=3,
        zero_division=0,
    )
)

print(
    confusion_matrix(
        y_true,
        y_pred,
    )
)

/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

  Irrelevant      0.798     0.878     0.836       597
  Complement      0.160     0.121     0.138        33
  Substitute      0.000     0.000     0.000       159
       Exact      0.756     0.896     0.820       578

    accuracy                          0.765      1367
   macro avg      0.428     0.474     0.448      1367
weighted avg      0.672     0.765     0.715      1367

[[524   5   0  68]
 [ 11   4   0  18]
 [ 66  12   0  81]
 [ 56   4   0 518]]
